# Pratica — Lezione 6: cambia la "personalità" del tuo chatbot

## Contesto

Il vero `lab-instructions.md` del materiale sorgente (lo stesso che hai usato per
costruire il chatbot di SmolLM2 nella lezione) suggerisce testualmente di
sperimentare con il **system prompt**, la **temperature** e il **top_p**
per cambiare come si comporta il bot — per esempio, rendendolo "più amichevole"
cambiando il messaggio di sistema e alzando `temperature` a 0.9 e `top_p` a
0.95.

In questa pratica farai esattamente questo: creare almeno una
"personalità" nuova per il chatbot di `HuggingFaceTB/SmolLM2-360M-Instruct`
e confrontare le sue risposte con quelle del bot con la configurazione
predefinita, usando le stesse domande di prova.

## Passo 0: installare le dipendenze e caricare il modello (come nella lezione)

In [ ]:
!pip install transformers==4.41.2 torch==2.2.2 accelerate==0.30.1 numpy==1.26.4

from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import warnings

warnings.filterwarnings("ignore")

model_name = "HuggingFaceTB/SmolLM2-360M-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.unk_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="cpu",
    torch_dtype=torch.float32
)

print("Modello caricato.")

## Impalcatura: funzione per una singola risposta

Per confrontare rapidamente le "personalità", questa funzione costruisce una
conversazione di un solo turno (system + user) con i parametri che le
passi, e restituisce la risposta del bot. Così non hai bisogno di un ciclo
interattivo `while True` per provare più configurazioni di seguito.

In [ ]:
def generar_respuesta(mensaje_usuario, system_prompt, temperature=0.5, top_p=0.8, max_new_tokens=60):
    """Genera una risposta di SmolLM2 per un unico turno di conversazione,
    dati un system_prompt e i parametri di campionamento (temperature, top_p)."""
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": mensaje_usuario}
    ]

    tokenized = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
        max_length=512
    )

    with torch.inference_mode():
        outputs = model.generate(
            tokenized["input_ids"],
            attention_mask=tokenized["attention_mask"],
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_p=top_p,
            do_sample=True,
            repetition_penalty=1.3,
            no_repeat_ngram_size=3,
            pad_token_id=tokenizer.pad_token_id
        )

    return tokenizer.decode(
        outputs[0][tokenized["input_ids"].shape[-1]:],
        skip_special_tokens=True
    )


# Domande di prova: le useremo con personalità diverse
preguntas_de_prueba = [
    "What should I have for dinner tonight?",
    "Tell me about your day."
]

## Personalità predefinita (linea di base)

Questa è la configurazione che hai usato nella lezione: un assistente utile e
conciso, con `temperature=0.5` e `top_p=0.8`. Esegui questa cella per avere
un punto di confronto.

In [ ]:
system_prompt_base = "You are a helpful AI assistant. Give short and concise answers in 2-3 lines."

print("=== Personalità BASE ===\n")
for pregunta in preguntas_de_prueba:
    respuesta = generar_respuesta(pregunta, system_prompt_base, temperature=0.5, top_p=0.8)
    print(f"> {pregunta}\nBot: {respuesta}\n")

## Esercizio: crea la tua personalità

**Consegna**: definisci un nuovo `system_prompt` che dia al bot una
personalità chiaramente diversa da quella base (per esempio: pirata, maggiordomo
estremamente formale, allenatore sportivo iperentusiasta, poeta
malinconico...). Scegli anche i tuoi valori di `temperature` e
`top_p` — ricorda che valori più alti (per es. 0.9 / 0.95, come suggerisce il
materiale sorgente) producono risposte più variegate e creative, mentre
valori più bassi le rendono più prevedibili.

Genera le risposte per le stesse `preguntas_de_prueba` con la tua configurazione
e confrontale con la personalità base sopra.

Suggerimenti:
- Riutilizza `generar_respuesta(mensaje, system_prompt, temperature, top_p)`.
- Non serve ricaricare il modello: è già caricato in memoria.

In [ ]:
# TODO: scrivi un system prompt che dia al bot una personalità diversa
mi_system_prompt = None  # TODO

# TODO: scegli i tuoi valori di temperature e top_p
mi_temperature = None  # TODO
mi_top_p = None  # TODO

# TODO: genera e mostra le risposte per preguntas_de_prueba
# usando la tua configurazione, come è stato fatto con la personalità base

<details>
<summary>💡 Vedi soluzione</summary>

```python
# Esempio di personalità: un pirata molto espressivo, con risposte più
# creative (temperature e top_p più alti, come suggerisce il
# vero lab-instructions.md del materiale sorgente)
mi_system_prompt = (
    "You are a cheerful pirate captain. Answer every question in pirate "
    "speak, full of enthusiasm, in 2-3 lines."
)
mi_temperature = 0.9
mi_top_p = 0.95

print("=== La mia personalità ===\n")
for pregunta in preguntas_de_prueba:
    respuesta = generar_respuesta(
        pregunta,
        mi_system_prompt,
        temperature=mi_temperature,
        top_p=mi_top_p
    )
    print(f"> {pregunta}\nBot: {respuesta}\n")
```

Confrontando con la personalità base dovresti notare due effetti
distinti e sovrapposti:

1. **Il tono cambia** per via del `system_prompt` (formale/conciso vs. pirata
   entusiasta) — questo dipende quasi interamente dal prompt.
2. **La varietà delle risposte cambia** per via di `temperature` e `top_p` — con
   valori più alti, esegui di nuovo la stessa domanda più volte e
   osserva come le risposte cambiano più da un'esecuzione all'altra.

Con un modello così piccolo (360M parametri) non aspettarti un'interpretazione
perfetta del personaggio in tutte le risposte — è normale che a volte
"esca dal personaggio". È una limitazione reale del modello, non un errore
del tuo codice.
</details>

## Per continuare

- Torna al notebook di lezione:
  [clase06_llms_chatbot_huggingface.ipynb](../notebooks_clase/clase06_llms_chatbot_huggingface_IT.ipynb).
- Ripassa la teoria in
  [manual/09_llms_gpt_llama.md](../manual/09_llms_gpt_llama_IT.md) e
  [manual/10_chatbot_huggingface.md](../manual/10_chatbot_huggingface_IT.md).
- Nella prossima lezione esporrai un chatbot come questo attraverso un
  server web con Flask, in modo da poterci parlare da una pagina
  HTML invece che dal terminale.